# 00 - Dataset Atlas for created_frame.hdf

This first notebook establishes the chemical and materials-science map of
`created_frame.hdf`. The focus is not merely technical schema inspection,
but a first scientific answer to three PhD-level questions:

1. Which classes of calculations are present?
2. Which materials and surfaces dominate the database?
3. Is the dataset chemically rich enough for adsorption and pathway analysis?

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Read the OnePiece-style HDF table

In [ ]:
print("rows, columns:", df.shape)
df[["Name", "Formula", "E", "fmax", "record_class", "facet", "material_family"]].head(12)

## 2. Schema and completeness

In [ ]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [int(df[c].notna().sum()) for c in df.columns],
    "fraction_non_null": [df[c].notna().mean() for c in df.columns],
})
schema.sort_values(["fraction_non_null", "column"], ascending=[False, True])

## 3. What kinds of calculations are present?

In [ ]:
class_counts = df["record_class"].value_counts().sort_values(ascending=False)
class_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
class_counts.plot(kind="bar", ax=ax, color="#4c78a8")
ax.set_title("Database composition by record class")
ax.set_ylabel("number of calculations")
ax.set_xlabel("record class")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Which surface families and facets dominate the database?

In [ ]:
surface_like = df[df["record_class"].isin(["clean_surface", "adsorbate", "copt"])].copy()
facet_counts = surface_like["facet"].replace("", np.nan).dropna().value_counts().sort_index()
material_counts = surface_like["material_family"].value_counts().head(20)
display(facet_counts.to_frame("rows"))
display(material_counts.to_frame("rows"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
facet_counts.plot(kind="bar", ax=axes[0], color="#72b7b2")
axes[0].set_title("Facet distribution")
axes[0].set_xlabel("Miller index")
axes[0].set_ylabel("rows")
material_counts.sort_values().plot(kind="barh", ax=axes[1], color="#f58518")
axes[1].set_title("Most frequent material families")
axes[1].set_xlabel("rows")
plt.tight_layout()
plt.show()

## 5. Chemical formula landscape

In [ ]:
top_formulas = df["Formula"].astype(str).value_counts().head(25)
top_formulas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
top_formulas.sort_values().plot(kind="barh", ax=ax, color="#54a24b")
ax.set_title("Most frequent formulas in created_frame.hdf")
ax.set_xlabel("rows")
ax.set_ylabel("formula")
plt.tight_layout()
plt.show()

## 6. Gas references already contained in the dataset

In [ ]:
gas_refs = df[df["record_class"].eq("gas")][["Name", "Formula", "E", "Path"]].sort_values("Name")
gas_refs.head(30)

## 7. Thesis-level interpretation

This atlas already shows that `created_frame.hdf` is not a narrow single-project
table. It is a chemically heterogeneous research database containing:

- bulk oxides and metals,
- clean surfaces across multiple Miller indices,
- adsorbates ranging from CO and CH3O to HCOO and COOH,
- constrained-optimization (`copt`) pathways,
- gas-phase references needed for thermochemical reaction conventions.

In other words, the file is large enough to support a dissertation-style
discussion of structure, convergence, adsorption chemistry and reaction paths.